# Laya multilingual FP32 기준 모델 v3 — laya-edge-bench README 선택지

| 판단 작업 | 선택지 |
|---|---|
| 상태 판단 (`diagnosis`) | 정상 · 센서값 이상 · 데이터 수신 중단 |
| 대응 판단 (`policy`) | 계속 관측 · 경고 발생 · 연결 점검 요청 |

**정답 규칙** (온도 센서 예시, 1초 발행, Keep Alive 10초)
- 끊긴 지 15초 이상(브로커 LWT 수신) → 데이터 수신 중단 / 연결 점검 요청
- 끊긴 지 3~15초 → 데이터 수신 중단 / 마지막 온도 80°C 이상이면 경고 발생, 아니면 계속 관측
- 메시지 정상 수신, 온도 80°C 이상 → 센서값 이상 / 경고 발생
- 메시지 정상 수신, 60초 상승 5°C 이상 → 센서값 이상 / 계속 관측
- 그 외 → 정상 / 계속 관측

**위험 오답**도 따로 센다: 이상 상태를 정상으로 본 경우, 경고·점검이 필요한데 계속 관측으로 답한 경우.

**Colab**: 런타임 → 런타임 유형 변경 → **T4 GPU** → 런타임 → **모두 실행**

**Kaggle**: Accelerator **GPU T4**, Internet **On** → **Run All**

학습·검증·시험 데이터는 노트북 안에 들어 있다 (웹 목업 시뮬레이터와 같은 규칙으로 생성, 시드 대역 분리).
결과는 작업 폴더(Kaggle은 `/kaggle/working/`, Colab은 `/content/`)에 남는다: `laya-fp32/` 체크포인트, `results_rows.csv`, `results_table.csv`, `metrics.json`, `laya-fp32.zip`.

In [1]:
import torch
assert torch.cuda.is_available(), "GPU가 안 잡혔어요. 런타임 유형을 T4 GPU로 바꾸고 다시 실행하세요."
print(torch.cuda.get_device_name(0))

Tesla T4


In [2]:
!pip install -q "laya==0.4.31" scikit-learn tabulate

In [3]:
import os
os.environ["USE_TF"] = "0"              # transformers가 TensorFlow를 건드리다 멈추는 문제 방지 (Laya README)
os.environ["LAYA_CUDA_AMP"] = "float32"  # 추론을 FP32로 고정 (기본값은 반정밀도)

import base64, gzip, json, time, statistics
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
DATA = {
    "train": "...",
    "validation": "H4sIAP06/GoC/+3dbW8c1RXA8ff9FKt91Urxau6dmXtn8mUs19nQlRw7eDdQhCKFxkU8BDVADAYcMFIQqUQlNwnCSOmbfhzv+Dt0ZuNAHs4J3mbWwWf+fVNs7N2N1/lxzj33nPN2fzxZmQz759/uXxi+MVodLo8u9M/3V4dra0uJ65/rT4aXLi+v9s/n2aA8/ujCcG2yshyScfP5pWQQzvXHo/X6W9dWxpPlS+PXlsf988kgO9e/9Ppksrz25mR54+LFtdF6/TQXV9bGw3P9N0frFzbebL4uJFfP9V+/MhxPRhvr49nrGK28tr4xHs0+mLx1edi8nr9s1K+tfjmj9fFk88rq8Rf3qx93D3962Ku2HlZbu4f7nxzerz94b6f6cK93eLA//fZ29fnN+t/0qs8eTD+706uuv3N0vf66a73DB9eqg2+nN3emH9yqvq8/vr83/e5BtXVQfXlrUD/P6uZoMtwcrTSvodrbrr+xfrbpP29VH+7WX948QrW3O92+0Zt+eqd+8GrnTv0IT7+O6f1rR1/cmn7zfnX7Qe/RY/Sm925Vu1v14//ylb363z7z6PVL+vVhm4+/v/HrIzcPNn33k8N/70//ttOb/ty87MN7N2bPtn23fuTpR/v11xxt7T/+QVR3Pp5+eLd5gu/fmz3a+71fnqp59qOvtnr1R4f7W81jHx4c1E/cO9rZqr7+oTd7wq3e0Rfb1e2Do+295jF71efvVrdv9K/Wb9zljbXR6lsneKMe/7x3j9+D4/elfuXNK/jgoPrqbv0Ps5/ro7dqeuNadfvj2ZNvP6i2H9b/d5L36vD+VvXuR8dvb/MLUr9ZB+9V93am3zw8/mzzmM0P/bP66W7W33547z/Nezfd362u326+5cv65/5O9fXN+jU2P9lH/376XfOOPphen717n+8f3qt/wHv1F9Sv+Mtb1b1/Nd85e5N6078fTD/drb641ptu3Zn92W4cvfND76lvmr2I2fc179rV5mc5/Ovl4epkeOGZvwOPf/1+/WE/84esv/XScDL70483rmyuNm/DxeHK5Mpm8z6ML6+NJvVn3lhZG11Yad6O+pN/3riyfmFl863jv49Xr/7h7ZM6EPJBITng/MDJDgQcwAEceHkHnv1V+b2IUA68GBm4QZREcC4dpJIJ9e8JJEACJJyYBO0NfUIG8fWcLhC5EjIs+SZHkEKGkpABH/DBWOrgklQ+Q/BNACEECoMcB3AABxabOjz12k9XhKJsUgQxdcglEXIiA0RAhNPJHF4lDDHIZwppk0k874KXDxRwARdw4feTMTQHfHOFB0rNUYkOHCVHFEABw/lCyJVLCK75/PMipJE6Ax7gQTfqDDFINiRKFXIQCRbAARxspQwhkQ8OnNNqDJwcwAAM2L2e5JJEDAvEfCGVbyuSMaABGhjMGNxAyRlyggV4gIduXEyKTkIglyuNDgIgAALOSr4wd81x9h95scQg9jm6rLmSgAiIgAjG7yQV8tFiJhcYiBRwARfMFRji7JaRUGDImriBJgYYgIFuFRjKRBYhyAlDAARAAAT7+UJItNvLyk0E8gVcwAVzXc+uHIj1hZRxSSiAAh3sec7kIQjyMWLgShIgAEJXriRlmcyDcrzoiRbAARyM9T0r1xLlCwgDDMAADLCbMeRBDggKeboqJwh4gAenVF94hQeLeVCGIXhtIwPdC8AADNYKDNoQRadNPkABFEABu0OSvBwXBLmpsSQsAARA6EC+oKxh8PLsNLYw4AIumKstlJmcLfhE3t/mGaoKAzBwdvKF/6PYWMqj1sWwwOXcR8IDPDij95HmbncsmlMDqYmBpAEd0KEjSUOqKJArQ1EYrgwDMGA3aYhRO0v0yqUkLikCAiCctSLD3OPXS/nmciofLpIuwAIs2LuSFJSriU7d9Ex4gAM4YHdIUn1JURQhV6eqck0RERDB/pSkIJ8kLCWDlJQBGIChI10MTi4xBPnkgF3PMAADptsYCm18ojwIRRm0zL0kQAAEe3OSci/fY061MeycJ8ADPJhLGxLlIoLXHKDbEQdwwOzNpNLNNXXdJ6QNeIAHnRmvWsrxQtqcJzKOHR7goRNpQy7fU8y1tiYYgAEYsFttcM7J54naPQSfy3ufSRwgARLsJQ4h0xqhoxwxsBUeHuDBWuIQgzJslSlqOIADHexsyIIcGHj5RNFl7H9GBETowLzV2VUkcT+DPJedQ0ZgAAZzKUNeaJ0NGZ0NMAADZzxjmPeGUq5seC1kDzI8wAM86EC+kDRXFJ93IdHiBNIFXMAFa+mCc6VybuDkBU4kDEAABJYvJ0WlFTpTQOBqEiAAQldWNOTalQRtcgqzFtEBHazlDeVsZKLAgNb5iAM4gAOGF7sVMghaR4PDAzzAA/MTV6NXAgUtX6DOgAu4YO5aUiKXGepOhsC5AQ7gQOc6GVLtIFEEISSUGQABELrSAR21CQlqCzQBAz7gg7lCg5dnJ2XkDTAAAx3c1JBow9TkA8XIyFVAAISOXE8KTm53UtsaGJyEDuhgLmtIlLnsTpuGQNqAAzhgOG0I8+xvyuUuJ5IGOIADc0lD3fGkbHfLCRbQAR3OZNIw9y6nXLmpmMgni7Q0wAAMWF7uFuU7i745NBRnrbLtERAAwXxPQ+7kEuSSl5sfkya9AAZgAAZTVYYgLneMDFwGARDoXkdD9PK2V+38oJTzBYoMgAAIBpdBp9pOt4LjRXiAh65UGQp5P0OQjw7Y34QCKGC2yJCV8hmCuN8xVbocSRnQAA0MroHO5ZQhU7bENwcNBAvwAA+mUgYnJwxKmYFaIwZggOFOhlzubVImqdVlBhqgAQEQOpIz5F4ZnJRoJ4zcS8AHfDDXAp3K7QzeyWeLjkoDDuCA3UqDS5ycOChN0JHTREAAhA6saFD7nDI5YaDNCRdwwViVoXByrXEp0QoNhAcwAAOGCw2lLIJyOcmlkalJiIAInWloSOQTRqXQQLgADuBgrswQmivJUpCQcUURBmCga/0Mylq3qMxSDNxOwgM86Mik1ayU1zN4+UiBu0nogA72koZC2eHknbbVDQdwAAfMZg1FKV9NypU+aO4q4gEenLmrSXMvZlDShUwemZTI9UdYgAVYsLeYIdXuJ7LTEQVQwG6ykCnj1xMlW/DUGAABEDpzLykop4vyRWZPvAAP8GAta3BJIrc1lfI4drY6wgAMWO5nKAo1bxCPE4smnUAEREAE4w3QhVxmWHJyqMBQdmAABnsd0IXcyuCidjuR+AAGYMBuxqBdV9YyBpfLFxWpNCACIpjrZohB3uWUaZNWiRfQAR2sFRqcMiclVVoeKTTAAAxYThtSudBQKoOTkigvbSFtQAREMLjWTel2cvIgZlY6wQM82OuCrje1pHLTo7IQnrnLMAADC84bXuWNReUgoZQbnVyGCIiACJ3Y0ODlQoOjowEXcKEThYaoTE0KSnhAdIACKGA3Xyij5oGXLydl3E4CBEDozCboRM4avHwvga4GeIAHg2UGZXxSzh5oGICBzqUNeSlvaAjKfoaCnAEN0KAb+xmcfKTgtFGLxArogA7m9jMUygXFSEsDDuBAB3dBR7nVsVBuIOSkDYAACJ0ZuapcUHLaGSNbYOEBHoxdUKpPFuXTA20jPAzAAAycnVLD/AcJQas95nQ0IAIidHexm9LQkNPQAAuw0I3Fbsp4xVJb78j8NBRAAbtVBq2jIVM6nCKL3QABEDpyOSl4OWlY8loDJMeL8AAP1m4nRScfKip9j57xaTAAA4b7oJU+x6BUHdOCy0mAAAhduZwUc2W5m7LViRIkOqCDtayhVHIGpdRAwREEQMBwqSHThidFMWlgCDsiIIL9m0m5MoPZaUOTAizAAixYG5qktDKoq6AZhAADMGC3kyFP5aJjqnQ4FvJIBGoMgAAIFmsMctrgtXiB80V4gAdzg5O0kcvqLmgcwAEcODN1hnmPE4tEnpuk1B1d2lxlBgRAAATrDdBujnzBc74IC7BgrQG6SOQN0N5pHU1UG2EABsymC/WYVXE3g7LQTU4WqDGgARrY282QK+3P6nRFbizCAzyYa3/OtHZHpf2ZE0UYgAGz7c+Fl5e1LIkc1EvkSRrgAA66sgQ6laOFlN2v4AAO3ZizmjMwCQVQgCrDcUzg5LnLS7NypLgEuiRnAARA6EShoT5gjCc/UHDMSwIHcLA3LymRk4ZSThrogIYBGLA8MSlmcotjoUxMylgKjwiIcEqtDK9yNEImHyck8iXmhONFXMAFcwlDUHa91oGAEwMEFEABFLC7+zmVW5+XvHyC4KIcGFBlQAREsNfO4OXjhEQ+TmD5Kzqgg8llbuL9xFwbnEY3AwzAgNluBpcU8uTlRD5FyIgLAAEQztzApPmrDFFWgdsIqIAKv6tsYbw6XF/ZHG00Cjz+x8eP1Gv+gHvNH2M8bJ4tT5r/1X+1Hw05eR4NjMAIjMCIR0ZEjMAIjMCIFxjhWggkgjxTQTmEgAmYgImzxgSxBEiABEi88EyilVhCvAeVgARIgIQFJGIbSARlJAPBBE7ghAUnUg4mYAImYOI3mOBgAiRAAiRehETWxm0JeftUIu+qhQmYgImzxgSxBEiABEi8CImcIgdIgARIvBCJuLDDyyVOL3ECJ2w4Efwii6EZTuAETlhwopV4Ilec4AQTJ3DCRMdXK4UOxxkmTMCEYSbiApngdAImYMICEwVXMGECJmDiN5hoJZrwMhMBJmACJgwwUfoFMpHCBEzAhAUm4gLvasMETMCEiYE0yQJ7OlACJVDChBILa+nACIzACAtGOGZpowRKoMSLlWghkkiVbT0ogRIocQaUeOaX7o/Vzt3pP7aat6T64Oc/PemGa29PR6bt/ixxAzdww6QbsQ03HBNrcAM3uuRGG4cZmcwG7aSwARtG2YiwARuwARvznW540hTgAA7gmBeOVqopkdYy4ACOLsGRthJxKKlKARzAARw24Wgh4sjlSqyjEgscwGEUjmxxUzBSGtKAAziswtFCxFHIqUpGqgIcwGEUjjZWjJQyHDlwAAdwLBCOZ39VXhEhLx97hESur0TqKxACIeYJaWE3SfAyINzsABAAMQ9ICzGI0jFbQgiEQIh5QlrYZxJyeWaoc0znwBAMsW9IC2FIkI9CnG8+jyEYgiGmDWlhE0oolDgk4zwVQzDEviEtxCGlcgGVsi6GYIh9Q1rYoRIT+S6qC5ypYgiG2Dfk5eOQmMqLE1wkDsEQDDFvSBvbV2IuD/5wBYUZEAGRDiDSQiQSlBORglNVEAER+4i0MPA0FmokQnkXREDEPiItRCKlUt8tOFcFERCxj0gLw1ALOQ5hvBCAAMirAeSp175gQF4+Cim8eh4CIiACIuYRaWFAaqFsnHMl88pABEQ6gEgLkUhQ7oiUTUsviIAIiNhGpIXBqUVULquWVGZABEQ6gEgLkYhWmQEREAGRDiDSwkDVUqzMeJZNQQiEdIGQl49DSq8mM5yIgAiImEekhZGqZaaMIgIREAGRLiDSQiQSOBEBERDpLiItDFYto3pLJAMREAER84i0EIkotRmfQAiEQIh1QloYrFo/VjN1SAxEmEaEIihiX5HYgiJe3phZK0LnDIqgiHlFyjZikVSt8XIogiIoYl+Rl49F8ijPI0oYjIghGLJAQx79+rUxP+TJX4j//jS9/+Phjz886YU/9sK3oUUqa0HEgRZoYUmLVmILL2tBaz9aoIUlLVoYjZorXfxLCddN8QIvbHkR2/AiKl5wdoEXeGHq7MLjBV7gBV6c0Is2ji/kwwu68MECLCxhkbYSXIiXMbhbDhZgYQqLVk4ugpKJUBnBC7yw5EV2+icXGV7gBV604IX2hp4qIKdfKikBBEAAxAgg+elHII4QBEEQxIwgpx+COCUGqX+rAARAAKRNQMTXswBHwulHIj7DERzBEVuOtBKPFIoj9LLhCI6Y6mVbXNwBF3ABF8a4WNxxB/fBwAIsLGFR+AUO4SG2gAu4MMVFXNhd8yV6U/ACL2x5UXpm/MEFXMDFybhoJbzICS/wAi+6MIcnWejgLs4vAAMwbIGxuLsXBBh4gRfGvGhnMihe4AVedMIL4gu8wAu8OOngcb/IeTyAARiAYQuMhTazAgZgAIYpMFJOMPACL/DipF6oAcb/AMFoEy1o5QMA",
    "test": "H4sIAP06/GoC/+3dX28TVxrH8ft9FZavdiVizX+f4c1EUTBdSyGh2JStEBIsXkQh1dI2gUCTNpWoYCVWyoagphJ7sy/HM3kPe8aEvzqHSfowrvrMlxtix/PHHuejc35zznOudkfjpfGge/Zq99zgi+HyYHF4rnu2uzxYWVkIwu6Z7nhw4eLicvdsGvfy40fnBivjpcUsGFXPB2e6o+Gq3W5laTRevDD6bHFkn+ylZ7oXPh+PF1eujBfXzp9fGa7aY5xfWhkNznSvDFfPrV2pXpcF1850P788GI2Ha6uj2UkMlz5bXRsNZw/GX14cVCfz1zV7YvZchquj8aXLy8cv7pYvtqe/vOyUk5flZHu69+30uX1we6u8u9uZHu4VP+2UD+7Z33TK+wfF/ced8uaNo5v2ddc704Pr5eFPxb2t4s5G+cQ+fr5b/HxQTg7LRxs9e5zlS8Px4NJwqTqHcnfTbmiPVvxro7y7bV9e7aHc3S421zvFd4/tzsutx3YP759H8fz60cON4sevyp2Dzqt9dIr9jXJ7Yvf/5pUd+9sP9m5P6e1uq8dP1t/uudpZcevb6X/2ir9vdYpfq9Oe7q/Pjrb51O65+HrPvuZosvf6gygff1PcfVod4Mnt2d6+6rw5VHX0o+8nHftoujep9j09PLQH7hxtTcofnnVmB5x0jh5uljuHR5u71T475YNb5c5695q9cBfXVobLX57gQr3+vLePr8HxdbFnXp3BncPy+6f2h9nn+upSFevXy51vZgffPCg3X9r/TnKtps8n5a2vjy9v9QWxF+vwdrm/Vfz48vjZap/Vh37fHu6e3Xy6/9/q2hV72+XNnWqTR/Zzv1H+cM+eY/XJvvp98XN1RQ+Km7Or92Bvum8/4F37AnvGjzbK/X9XW84uUqf4x2Hx3Xb58HqnmDyevbf1oxvPOu9tNDuJ2XbVVbtWfZaDv10cLI8H5z74G3j99Xv7YX/wJu2mFwbj2bsfrV2+tFxdhtHyYHXp0nDNbvXmx9d76lRvcLd6G6NBdbQ8Dew/+4fdPRvZ5y6uDO1P9s98NLbn9aer6IAO6IAO3bOJTAfj0mEh6IUAARAAoQCITAJE4mw8oAM6oIMKHQw6oAM6oINThzCAB3iAB3hw8yCKJpNe7AYiAQiAAAgNQJBOIgRCIIRfCOJJeIAHePDwYGhAIARCIIR3dFTA8Ch4gAd4cPMQ0b+AB3iABzcPCa0HeIAHeHDzkMl46HvihwghEAIhNAhhEAIhEAIhfELEBJTwAA/w4OGBgBIe4AEePDwksiHWkRuIGCAAAiA0ACEbQNnztCBSgAAIgNAAhGyKtzuhDKqRlQABEADxx68gJZvkXf3Ju4DIAAIgAEIDEBFdDIAACIDwAZEABEAABED4gMgoFAMQAAEQPiAMQAAEQACEB4hUGFJmbiD6AAEQAKEBCGpRAgRAAIQXCEJKgAAIgPACQUgJEAABEF4gCCkBAiAAwrvUXkAXAyAAAiB8QEQMtQYIgAAIHxDCkNJZ755qEPAADyp4yBrggbmcAAEQSoAwjXQwSCAAAiA0ANEPAAIgAAIgfEA0E1FSzxogAEIFEEkjQDATAyAAQgUQhJQAARAA4QXC0IIACIAACA8QJqDoPUAABED4gIgaAYKQEiAAQgUQCeViAAIgAMIHRNbAqhgMtIYHeFDBg4EHeIAHeHDykDPRGyAAAiC8QDSzZA6lYgACIFQAIQwoQxcQC0GPPgZCIIQKIbJmhGAUBEIghAohRCFl3Ms9QjDSGiEQQoMQYRDIiHAOtV6gkwEQAKEDiEgGROwGghwCIiBCCREJREAERECEn4hMRkTiSSJyiIAIiFBBhDCszDxEMCUDIiBCBRFhM2ElHQ2IgAglRES0IiACIiDCT0TSzKgIJmdABEToIEIYVxqIgAiIUE2EKK5M3RWmFigxBREQ8ccg4oMv3Z/LrafFPyfVJSnv/PqXd9EIj9GIZGS453ziBV7ghUovEpkXnjLY3O1ADMTQKUZGpwQyIAMyTk6GMMdgFjlkQEa7yAiF1e24g4oZmNEyM2Thp0cM6t0hBmIoFSOhlYEZmIEZpzAjwwzMwAzMOIUZjOTCDMzAjNOM5AowAzMwAzNOYUZE3wQzMAMzTmEGGShmYAZmnMaMJlYupgQGZmCGXjMM7QzMwAzMOLkZccDIcczADMw4hRkR900wAzMw4xRmJJTNAAzAAIyTgiGcBd93182gXwIZkKGUDAMZkAEZkHHy4lyi+DNzV+diLRLIgAy1ZESQARmQARknJyORkWHcZFA3AzIgQykZoviz777HGlaSQAZkQIZGMoyMDOeqiRGjMiADMpSSkYriT+NuZUS0MiADMrSSEcnIcGYZMVkGZECGVjJE8WfuHvsJGIABGErBkISfWeCelJYw9x0yIEMrGUZGhnPue8rUd8iAjAbJ+PCr8jstuCiJQbPQPXA8Aw/wAA/9eEgC0Sxy30PJuIcCHuChH49EhodzMDk5B3RAh3o6RCFp7ISDVgd0QId+OkRhadxz4mEYKQoe4KEej34gw8O48cjBAzzAQzseorA0cc9RyaunwQM8wEM3HokMD+PGg5YHeICHejxEcamnAHHI4ijogR4t0MPI9OijB3qgR0v1MAF6oAd6oMdv0kOUmWaeUmAhdc3RAz306yEKTfvuubQhZcHQAz1aoEcm0yPz6ME9F/RAD/V6iFJT4x5nGlIsDD3QQ78eeSDTI/fowRwX9EAP9XqIUtPc03NJaHugB3ro10OSmvYDT88lIfdAD/TQr4ckNe2777ik3K/FDuzQb4cR2eHpt6S0PNADPdTrMfv/t/MReTouKaEpfMBHC/iIZHy4V6rO6LvAB3y0gA9RbBp7Oi8UQ4YP+GgDH6LcNPF0XmZj2OEDPuBDOR+i6DT14MGAD/AAD/V4hKLgNPUkH31KE8IHfLSAD1FwmnnKi8EHfMBHG/gQBae+Wfp9lryGD/hoAR+i4NTXdWHIGHiAh348RLGp6aV0XeADPtrKRxTI+Mg9fFBgDD7gQz8fouA097Y+GHEKH/Chnw9JcGoCb3BK5wU+4OP34eO9c2+Yj0zGhy86ZcA6fMCHfj4k0akJPQPWwQM8wEM9HrEkODXuVV0MsSl4gEcL8IhEeHiWlDNMlYMP+GgBH6LYNKbtAR7g0Vo8RKFp7G17cMsWPuBDPx+i0DTxTJUzzHWBD/jQz0cSyPhwjzfN6bzAB3y0gA9RcJp6xpvmdF7gAz5awIcoOM08XRdm2oIHeOjHQxScZp62B3zAB3y0gQ9RcDqbUQsf8AEf7eQjDWR8+G7bMlwdPuBDPx+i4NR4+MgZcQof8NECPhIZH7mn9cFEffiAD/18iKLT3DNRn84LfMBHG/iQRKe5r0zQrPYpfMAHfOjmIwtkfBj4gA/4aC0fkug0D70zXsg+4AM+9PMhiU7ziM4LfMBHi/nIZHxw5wU+4KO9fIii09hX45QJc/ABH/r56AcyPnIKFcIHfLSWD1F0mninzMEHfMCHfj4SGR99loeCD/hoLR+i6DTtZfABH/DRWj5E0am33gd4gAd4aMfDiILTvve+SwIf8AEf6vmIZHwYbtvCB3y0lg9RcGp6LBEFH/DRXj5EwWnuLVVI5wU+4EM/H0bGR0rrAz7go6185IGMj9xz25ZCyfABH/r5kESndi++SsnogR7ooV6PRKRH6O28MF0fP/BDvx+ZyA9/uQ+yU/zAD/1+SLLT1FOrMMAO7MCOxux49fV7V4z33uQpxHj3C/G/X4rnL6Yvnr3rRHTsRIQSKIESKPFRJRKZEs4gdCFg9j1QAIUqKDIZFJkHCgZsAAVQaIJCmE4kHigIN4ECKDRBEQYyKWKPFNxGRQqkUCVFREqBFEiBFLVSCPPM1CMFk+KRAilUSSEMNCOPFFT+QwqkUCWFQQqkQAqkqB1yFSAFUiAFUtRKEXGXFCmQAilqpUiQAimQAilqpciQAimQAilqpTCMvEIKpECKOilixmgiBVIgRb0UjNFECqRAinopEipTwARMwMTHmRDGmc4a37QngAIolEFhgAIogAIo6mpdBXQ8YAImYOLjTFA5EyZgAiZqmBDFmKG7ij8zzYECKJRBQZAJFEABFLVQmDkOoQiRAimQ4o8pRRrMUYoYKZACKT6BFL4LOlc65jlOM4UO6IAOLXTMc7GgPnRAB3RooSObIx05dEAHdGihY66pKLEodmCHFjuyeeakIUEpdmCHGjvmGZSGnqTUfp+gAzqg41PS4TyfJgSZZ14a9hEEQRBElyDzjE3DHEEQBEF0CTLP9DQKEQRBEESVIP15ZqhRjCAIgiC6BJlnkhqRpCIIgigTZJ5JakSSiiAIokyQrJFqHTmT8AEEQBRNre0bGRTOFU9CqnUABVCogsIEMiici73DBEzAhComogaYmBUZBQqgAAo9UDRTT7QPFEABFJqgkEWZbiYymIAJmNDERBMjP4NeAhRAARSKoMiDJu54wARMwIQmJqIGuh2sdgITMKGKiYTVToACKICiDoomYkyYgAmYUMVEQxPYI6RACqRQJEUYBA20KHACJ3BClxMRTuAETuBEjRPJJ3digYgCKIBCGxTNTC6nSYEUSKFLCkOTAiiAAihqoAiDBoZRcNsDKqBCHRVRE70PnMAJnFDlREKTAiqgAirqqcigAiqgAirqqRCOz0w8VFC3BiqgQhUVkTDTTD1UULsGKqBCFxUNLRlE9RqogApdVBBrQgVUQMUJqBDGmrGHCopxQwVU6KLCkFVABVRARS0VcUCrAiqgAirqqYigAiqgAirqqUjogEAFVEBFPRXCWDPzUJFCBVRAhSoqHLHm/wFvMsr3pLYDAA=="
}
for name, b64 in DATA.items():
    with open(f"{WORK}/{name}.jsonl", "wb") as f:
        f.write(gzip.decompress(base64.b64decode(b64)))
def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]
rows = {k: read_jsonl(f"{WORK}/{k}.jsonl") for k in DATA}
QUESTIONS = rows["train"][0]["questions"]
print({k: len(v) for k, v in rows.items()})
print(json.dumps(rows["test"][0]["state"], ensure_ascii=False))

{'train': 1081, 'validation': 242, 'test': 227}
{"device_id": "cell-01", "temp_c": 53.9, "temp_delta_60s_c": 0, "since_last_msg_s": 0.5, "mqtt_lwt_offline": false, "window_s": 60}


## 1. 평가 함수
진단·정책을 **한 번의 호출**로 같이 받는다 (같은 모델에서 두 판단). 정확도, Macro-F1, 출력 적합률, 지연을 잰다.

In [4]:
import torch, laya
from sklearn.metrics import accuracy_score, f1_score

def evaluate(agent, data, warmup=5):
    for r in data[:warmup]:
        agent.predict(r["state"], QUESTIONS)
    out = []
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for r in data:
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.perf_counter()
        res = agent.predict(r["state"], QUESTIONS)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        ms = (time.perf_counter() - t0) * 1000
        a = res.get("answers", {})
        pd = (a.get("diagnosis") or {}).get("choice")
        pp = (a.get("policy") or {}).get("choice")
        out.append({**r.get("meta", {}), **r["state"],
                    "exp_diag": r["expected"]["diagnosis"], "pred_diag": pd,
                    "exp_act": r["expected"]["policy"], "pred_act": pp, "latency_ms": ms})
    return out

def metrics(out):
    m = {}
    for task, e, p in (("diagnosis", "exp_diag", "pred_diag"), ("policy", "exp_act", "pred_act")):
        keys = list(QUESTIONS[task]["criteria"])
        y = [o[e] for o in out]
        yh = [o[p] if o[p] in keys else "__invalid__" for o in out]
        m[task] = {"accuracy": round(accuracy_score(y, yh), 4),
                   "macro_f1": round(f1_score(y, yh, labels=keys, average="macro", zero_division=0), 4),
                   "valid_output_rate": round(sum(o[p] in keys for o in out) / len(out), 4)}
    # 위험 오답: 이상을 정상으로 보거나, 경고·점검이 필요한데 계속 관측으로 답한 경우
    miss_state = sum(o["exp_diag"] != "정상" and o["pred_diag"] == "정상" for o in out)
    under_act = sum(o["exp_act"] != "계속 관측" and o["pred_act"] == "계속 관측" for o in out)
    m["critical"] = {"state_missed": miss_state, "state_missed_of": sum(o["exp_diag"] != "정상" for o in out),
                     "response_under": under_act, "response_under_of": sum(o["exp_act"] != "계속 관측" for o in out)}
    lat = sorted(o["latency_ms"] for o in out)
    m["latency_ms"] = {"p50": round(statistics.median(lat), 2), "p95": round(lat[int(0.95 * (len(lat) - 1))], 2)}
    if torch.cuda.is_available():
        m["peak_gpu_mem_mb"] = round(torch.cuda.max_memory_allocated() / 2**20, 1)
    return m

## 2. 학습 전 (원본 Laya multilingual, zero-shot) — 비교용

In [5]:
base = laya.load("convaiinnovations/laya", subfolder="multilingual")
print("device:", base.device, "dtype:", getattr(base, "dtype", None))
base_val = metrics(evaluate(base, rows["validation"]))
print(json.dumps(base_val, ensure_ascii=False, indent=1))
del base; torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

device: cuda dtype: torch.float32
{
 "diagnosis": {
  "accuracy": 0.4628,
  "macro_f1": 0.2958,
  "valid_output_rate": 1.0
 },
 "policy": {
  "accuracy": 0.1322,
  "macro_f1": 0.0779,
  "valid_output_rate": 1.0
 },
 "critical": {
  "state_missed": 104,
  "state_missed_of": 134,
  "response_under": 0,
  "response_under_of": 93
 },
 "latency_ms": {
  "p50": 35.98,
  "p95": 37.1
 },
 "peak_gpu_mem_mb": 1272.3
}


## 3. 파인튜닝 → FP32로 저장
Laya의 `finetune()`은 원래 가중치를 **FP16으로 줄여서** 저장한다 (`laya/train.py` `save_checkpoint`).
과제는 FP32 기준 모델이 필요하므로 저장 함수만 가로채 FP32로 저장한다. 학습 자체는 FP32 가중치 + 자동 혼합정밀도로 돈다.
검증 데이터는 학습 전후 비교용으로만 넘기고, 시험 데이터는 이 단계에서 쓰지 않는다.

In [6]:
import safetensors.torch as st
_orig_save = st.save_file
def _save_fp32(tensors, filename, metadata=None):
    tensors = {k: (v.float() if v.is_floating_point() else v) for k, v in tensors.items()}
    return _orig_save(tensors, filename, metadata=metadata)
st.save_file = _save_fp32

from laya.train import finetune, TrainConfig
CKPT = f"{WORK}/laya-fp32-v3"
cfg = TrainConfig(epochs=10, micro_batch=8, grad_accum=4, encoder_lr=2.5e-5, head_lr=1e-4,
                  loss="rlcd", shuffle_options=("choice",), seed=0,
                  eval_data=f"{WORK}/validation.jsonl")
t0 = time.time()
summary = finetune(f"{WORK}/train.jsonl", "multilingual", CKPT, cfg)
print(f"학습 시간 {time.time() - t0:.0f}s")
print(json.dumps(summary, ensure_ascii=False, indent=1, default=str)[:3000])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

eval items 484 from /kaggle/working/validation.jsonl (skipped {}, 0 overlapping training/calibration items)
train items 1946, calibration items 216, eval items 484 (/kaggle/working/validation.jsonl), skipped {}, device cuda:0
optimizer budget: 1946 train item(s), effective batch up to 32 (micro_batch 8 x grad_accum 4), 610 optimizer updates over 10 epoch(s)


/usr/local/lib/python3.13/dist-packages/laya/train.py:724: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 1/10 step 100 loss 1.1305
epoch 1/10 step 200 loss 1.0576
epoch 1/10 mean loss 0.8410
epoch 2/10 step 100 loss -0.3871
epoch 2/10 step 200 loss -0.0223
epoch 2/10 mean loss 0.3502
epoch 3/10 step 100 loss 0.4174
epoch 3/10 step 200 loss -0.0000
epoch 3/10 mean loss 0.2815
epoch 4/10 step 100 loss 0.1814
epoch 4/10 step 200 loss -0.0000
epoch 4/10 mean loss 0.2835
epoch 5/10 step 100 loss -0.0000
epoch 5/10 step 200 loss -0.0000
epoch 5/10 mean loss 0.0269
epoch 6/10 step 100 loss -0.0000
epoch 6/10 step 200 loss -0.0000
epoch 6/10 mean loss 0.0031
epoch 7/10 step 100 loss -0.0000
epoch 7/10 step 200 loss -0.0000
epoch 7/10 mean loss 0.0000
epoch 8/10 step 100 loss -0.0000
epoch 8/10 step 200 loss -0.0000
epoch 8/10 mean loss 0.0000
epoch 9/10 step 100 loss -0.0000
epoch 9/10 step 200 loss -0.0000
epoch 9/10 mean loss 0.0000
epoch 10/10 step 100 loss -0.0000
epoch 10/10 step 200 loss -0.0000
epoch 10/10 mean loss 0.0000


/tmp/ipykernel_50/1798134179.py:14: RuntimeWarning: laya.train: choice calibration: the fit came back unchanged at 1.0 (accuracy 1.00, mean confidence 1.000 on the calibration items), which usually means they gave it nothing to correct; this type's confidences are uncalibrated
  summary = finetune(f"{WORK}/train.jsonl", "multilingual", CKPT, cfg)



=== Evaluation (before vs after fine-tuning on 484 held-out items from /kaggle/working/validation.jsonl) ===
Metric                 Before         After          Delta         
-----------------------------------------------------------------
Accuracy               0.2975         1.0000         +0.7025       
Loss                   1.6928         0.0000         -1.6928       
ECE                    0.4140         0.0000         -0.4140       
Brier (multi-class)    1.0229         0.0000         -1.0229       
Brier (top-1)          0.4499         0.0000         -0.4499       
Mean Confidence        0.6580         1.0000         +0.3420       
-----------------------------------------------------------------

학습 시간 589s
{
 "train_items": 1946,
 "calibration_items": 216,
 "eval_items": 484,
 "skipped": {},
 "epoch_loss": [
  0.8409693020044781,
  0.35024305370318526,
  0.28152175791195183,
  0.28346366519022537,
  0.026870179020601202,
  0.0031259548964864404,
  0.0,
  0.0,
  0.0,
  0.0

In [7]:
from safetensors import safe_open
with safe_open(f"{CKPT}/model.safetensors", "pt") as f:
    dtypes = {str(f.get_slice(k).get_dtype()) for k in f.keys()}
size_mb = os.path.getsize(f"{CKPT}/model.safetensors") / 2**20
print("저장된 가중치 dtype:", dtypes, f"/ 용량 {size_mb:.0f} MB")
assert dtypes <= {"F32", "I64", "BOOL"}, "FP32가 아닌 가중치가 있음"

저장된 가중치 dtype: {'F32'} / 용량 1228 MB


## 4. 학습 후 평가 (FP32) — 검증, 그다음 시험
시험 데이터는 세 시나리오(정상 운전 / 온도 상승 / 메시지 끊김·복구)를 처음 보는 시드로 한 번씩 재생한 것이다. **모델을 고른 뒤 마지막에 한 번만** 본다.

In [8]:
ft = laya.load(CKPT)
print("device:", ft.device, "dtype:", getattr(ft, "dtype", None))
val_out = evaluate(ft, rows["validation"]); val_m = metrics(val_out)
test_out = evaluate(ft, rows["test"]); test_m = metrics(test_out)
report = {"version": "v3",
          "base_zero_shot_validation": base_val, "fp32_validation": val_m, "fp32_test": test_m,
          "checkpoint_mb": round(size_mb, 1), "train_rows": len(rows["train"])}
json.dump(report, open(f"{WORK}/metrics.json", "w"), ensure_ascii=False, indent=1)
print(json.dumps(report, ensure_ascii=False, indent=1))

device: cuda dtype: torch.float32
{
 "version": "v3",
 "base_zero_shot_validation": {
  "diagnosis": {
   "accuracy": 0.4628,
   "macro_f1": 0.2958,
   "valid_output_rate": 1.0
  },
  "policy": {
   "accuracy": 0.1322,
   "macro_f1": 0.0779,
   "valid_output_rate": 1.0
  },
  "critical": {
   "state_missed": 104,
   "state_missed_of": 134,
   "response_under": 0,
   "response_under_of": 93
  },
  "latency_ms": {
   "p50": 35.98,
   "p95": 37.1
  },
  "peak_gpu_mem_mb": 1272.3
 },
 "fp32_validation": {
  "diagnosis": {
   "accuracy": 1.0,
   "macro_f1": 1.0,
   "valid_output_rate": 1.0
  },
  "policy": {
   "accuracy": 1.0,
   "macro_f1": 1.0,
   "valid_output_rate": 1.0
  },
  "critical": {
   "state_missed": 0,
   "state_missed_of": 134,
   "response_under": 0,
   "response_under_of": 93
  },
  "latency_ms": {
   "p50": 38.51,
   "p95": 39.4
  },
  "peak_gpu_mem_mb": 1281.7
 },
 "fp32_test": {
  "diagnosis": {
   "accuracy": 1.0,
   "macro_f1": 1.0,
   "valid_output_rate": 1.0
  },
  

## 5. 결과표 (과제 문서 형식)
시나리오를 기대 답이 같은 구간으로 나눠 한 줄씩 적는다. 실제 답은 그 구간의 최빈값, 정오는 맞은 개수/전체.

In [9]:
import pandas as pd
from collections import Counter
df = pd.DataFrame(test_out); df.to_csv(f"{WORK}/results_rows.csv", index=False, encoding="utf-8-sig")
table = []
for scen, g in df.groupby("scenario", sort=False):
    g = g.sort_values("t")
    seg = (g[["exp_diag", "exp_act"]].astype(str).agg("|".join, axis=1) !=
           g[["exp_diag", "exp_act"]].astype(str).agg("|".join, axis=1).shift()).cumsum()
    for _, s in g.groupby(seg):
        dok = int((s.pred_diag == s.exp_diag).sum()); aok = int((s.pred_act == s.exp_act).sum()); n = len(s)
        bad = int((~s.pred_diag.isin(QUESTIONS["diagnosis"]["criteria"]) | ~s.pred_act.isin(QUESTIONS["policy"]["criteria"])).sum())
        table.append({"시나리오": f"{scen} ({int(s.t.min())}~{int(s.t.max())}s)",
                      "기대 상태": s.exp_diag.iloc[0], "실제 상태": Counter(s.pred_diag).most_common(1)[0][0],
                      "상태 정오": f"{dok}/{n}" + (" ✓" if dok == n else " ✗"),
                      "기대 대응": s.exp_act.iloc[0], "실제 대응": Counter(s.pred_act).most_common(1)[0][0],
                      "대응 정오": f"{aok}/{n}" + (" ✓" if aok == n else " ✗"),
                      "출력 오류": bad,
                      "위험 오답": int(((s.exp_diag != "정상") & (s.pred_diag == "정상")).sum()
                                    + ((s.exp_act != "계속 관측") & (s.pred_act == "계속 관측")).sum()),
                      "추론 지연(ms, 중앙값)": round(s.latency_ms.median(), 1)})
tbl = pd.DataFrame(table); tbl.to_csv(f"{WORK}/results_table.csv", index=False, encoding="utf-8-sig")
print(tbl.to_markdown(index=False))

| 시나리오                          | 기대 상태        | 실제 상태        | 상태 정오   | 기대 대응      | 실제 대응      | 대응 정오   |   출력 오류 |   위험 오답 |   추론 지연(ms, 중앙값) |
|:----------------------------------|:-----------------|:-----------------|:------------|:---------------|:---------------|:------------|------------:|------------:|------------------------:|
| 정상 운전 (2~118s)                | 정상             | 정상             | 59/59 ✓     | 계속 관측      | 계속 관측      | 59/59 ✓     |           0 |           0 |                    38.9 |
| 센서값 이상(온도 상승) (2~58s)    | 정상             | 정상             | 29/29 ✓     | 계속 관측      | 계속 관측      | 29/29 ✓     |           0 |           0 |                    39.1 |
| 센서값 이상(온도 상승) (60~124s)  | 센서값 이상      | 센서값 이상      | 33/33 ✓     | 계속 관측      | 계속 관측      | 33/33 ✓     |           0 |           0 |                    39.3 |
| 센서값 이상(온도 상승) (126~198s) | 센서값 이상      | 센서값 이상      | 37/37 ✓     | 경고 발생      | 경고 발생      | 37/37 ✓     |           0 |           0 |         

In [10]:
for task, e, p in (("상태 판단", "exp_diag", "pred_diag"), ("대응 판단", "exp_act", "pred_act")):
    print(f"\n[{task}] 시험 혼동 행렬 (행=정답, 열=예측)")
    print(pd.crosstab(df[e], df[p]).to_markdown())
json.dump({"questions": QUESTIONS, "train_config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in vars(cfg).items()},
           "rules": "README 기준 v3 — 노트북 맨 위 정답 규칙 참고", "laya_version": "0.4.31"},
          open(f"{CKPT}/run_config.json", "w"), ensure_ascii=False, indent=1, default=str)


[상태 판단] 시험 혼동 행렬 (행=정답, 열=예측)
| exp_diag         |   데이터 수신 중단 |   센서값 이상 |   정상 |
|:-----------------|-------------------:|--------------:|-------:|
| 데이터 수신 중단 |                 13 |             0 |      0 |
| 센서값 이상      |                  0 |            70 |      0 |
| 정상             |                  0 |             0 |    144 |

[대응 판단] 시험 혼동 행렬 (행=정답, 열=예측)
| exp_act        |   경고 발생 |   계속 관측 |   연결 점검 요청 |
|:---------------|------------:|------------:|-----------------:|
| 경고 발생      |          37 |           0 |                0 |
| 계속 관측      |           0 |         183 |                0 |
| 연결 점검 요청 |           0 |           0 |                7 |


In [11]:
wrong = df[(df.pred_diag != df.exp_diag) | (df.pred_act != df.exp_act)]
print(f"오답 {len(wrong)}개"); wrong[["scenario", "t", "temp_c", "temp_delta_60s_c", "since_last_msg_s",
                                    "exp_diag", "pred_diag", "exp_act", "pred_act"]].head(30)

오답 0개


,scenario,t,temp_c,temp_delta_60s_c,since_last_msg_s,exp_diag,pred_diag,exp_act,pred_act


In [12]:
import shutil
shutil.make_archive(f"{WORK}/laya-fp32-v3", "zip", CKPT)
if os.path.isdir("/kaggle"):
    print("Kaggle: Output(/kaggle/working)에서 results_table.csv, metrics.json, results_rows.csv, laya-fp32-v3.zip 다운로드")
else:
    from google.colab import files  # Colab이면 브라우저로 바로 내려받기
    for fn in ("results_table.csv", "results_rows.csv", "metrics.json", "laya-fp32-v3.zip"):
        files.download(f"{WORK}/{fn}")

Kaggle: Output(/kaggle/working)에서 results_table.csv, metrics.json, results_rows.csv, laya-fp32-v3.zip 다운로드
